In [4]:
import sys
!{sys.executable} -m pip install tenacity

In [5]:
import os
import sys
from tenacity import (retry, stop_after_attempt, wait_random_exponential)
module_path = os.path.abspath(os.path.join("../.."))
if module_path not in sys.path:
    sys.path.append(module_path)
import json
import pandas as pd
from lib import utils
from lib import tokenizer as tk
import step1_prompt
import step2_prompt

In [6]:
_TOKENIZER = tk.Tokenizer("gpt-3.5-turbo")

In [7]:
'''def get_recommendations(data_frame, start, end):
    data_frame["Description"] = data_frame["Description"].apply(lambda x: str(x).replace("\n", " ").replace("<br>", " ").strip() + "\n")
    data_frame["Title"] = data_frame["Title"].apply(lambda x: str(x).replace("\n", " ").replace("<br>", " ").strip())
    data_frame = data_frame.iloc[start:end]
    data_frame["No."] = range(1, len(data_frame) + 1)
    data_frame = data_frame[["No.", "Id", "Title", "Description"]]
    return data_frame.to_markdown(index=False)'''


def get_recommendation_by_title(data_frame, title):
    data_frame = data_frame[data_frame["Title"] == title]
    # if len of data_frame     is 0 then return None
    if len(data_frame) == 0:
        return None
    return (data_frame.iloc[0]["Id"], data_frame.iloc[0]["Description"])


'''
def get_recommendations_filtered(data_frame, titles):
    data_frame["Description"] = data_frame["Description"].apply(lambda x: str(x).replace("\n", " ").replace("<br>", " ").strip() + "\n")
    data_frame["Title"] = data_frame["Title"].apply(lambda x: str(x).replace("\n", " ").replace("<br>", " ").strip())
    data_frame = data_frame[data_frame["Title"].isin(titles)]
    data_frame["No."] = range(1, len(data_frame) + 1)
    data_frame = data_frame[["No.", "Id", "Title", "Description"]]
    return data_frame.to_markdown(index=False)
'''
def get_recommendations(data_frame, start, end):
    # 创建数据框的副本而不是视图
    df = data_frame.copy()
    df.loc[:, "Description"] = df["Description"].apply(lambda x: str(x).replace("\n", " ").replace("<br>", " ").strip() + "\n")
    df.loc[:, "Title"] = df["Title"].apply(lambda x: str(x).replace("\n", " ").replace("<br>", " ").strip())
    df = df.iloc[start:end].reset_index(drop=True)
    df.loc[:, "No."] = range(1, len(df) + 1)
    return df[["No.", "Id", "Title", "Description"]].to_markdown(index=False)

def get_recommendations_filtered(data_frame, titles):
    # 创建数据框的副本而不是视图
    df = data_frame.copy()
    df.loc[:, "Description"] = df["Description"].apply(lambda x: str(x).replace("\n", " ").replace("<br>", " ").strip() + "\n")
    df.loc[:, "Title"] = df["Title"].apply(lambda x: str(x).replace("\n", " ").replace("<br>", " ").strip())
    df = df[df["Title"].isin(titles)].reset_index(drop=True)
    df.loc[:, "No."] = range(1, len(df) + 1)
    return df[["No.", "Id", "Title", "Description"]].to_markdown(index=False)

In [8]:
def count_tokens(messages):
    total = 0
    for message in messages:
        total += _TOKENIZER.count_tokens(message["content"])
    return total


def get_chat_messages(prompt):
    messages = []
    sys_msg = []
    if hasattr(prompt, "system_general_message") and prompt.system_general_message != "":
        sys_msg.append(prompt.system_general_message)
    if hasattr(prompt, "system_grounding_message") and prompt.system_grounding_message != "":
        sys_msg.append(prompt.system_grounding_message)
    if hasattr(prompt, "system_instruction_message") and prompt.system_instruction_message != "":
        sys_msg.append(prompt.system_instruction_message)
    if hasattr(prompt, "system_constraint_message") and prompt.system_constraint_message != "":
        sys_msg.append(prompt.system_constraint_message)
    if len(sys_msg) > 0:
        messages.append({"role": "system", "content": "\n".join(sys_msg)})
    if (hasattr(prompt, "user_example_message") and prompt.user_example_message != "") and (
        hasattr(prompt, "prompt_prefix") and prompt.prompt_prefix != ""
    ):
        messages.append(
            {
                "role": "user",
                "content": prompt.prompt_prefix + prompt.user_example_message,
            }
        )
    if hasattr(prompt, "system_response_message") and prompt.system_response_message != "":
        messages.append({"role": "assistant", "content": prompt.system_response_message})
    return messages


async def get_ai_response(prompt_list, user_input_list):
    for i, prompt in enumerate(prompt_list):
        prompt.messages = get_chat_messages(prompt) + [{"role": "user", "content": prompt.prompt_prefix + user_input_list[i]}]
        if prompt.optimize:
            if count_tokens(prompt.messages) < 12000:
                # print("Fall back to GPT3.5")
                prompt.deployment_id = "gpt35-1106"
    responses = await utils.call_openai(prompt_list)
    return [response.choices[0].message.content for response in responses]

In [9]:
async def mapping(user_query, data_frame):
    prompt_list = []
    user_input_list = []
    start = [x for x in range(0, 330, 50)]
    end = [x for x in range(50, 360, 50)]
    for i in range(len(start)):
        grounding = get_recommendations(data_frame, start[i], end[i])
        prompt = step1_prompt.Step1Prompt(grounding)
        user_input = user_query
        prompt_list.append(prompt)
        user_input_list.append(user_input)
    output_list = await get_ai_response(prompt_list, user_input_list)
    result = []
    for i, output in enumerate(output_list):
        output = json.loads(output)
        result.extend(output["output_list"])
    return result


async def validate(user_query, possible_recommendations):
    prompt_list = []
    user_input_list = []
    prompt = step2_prompt.Step2Prompt()
    user_input = "\n[QUERY START]\n" + user_query + " \n[QUERY END] \n"
    user_input += "\n[POSSIBLE RECOMMENDATIONS START]\n" + possible_recommendations + "\n[POSSIBLE RECOMMENDATIONS END]\n"
    prompt_list.append(prompt)
    user_input_list.append(user_input)
    output_list = await get_ai_response(prompt_list, user_input_list)
    return output_list[0]

In [10]:
async def main():
    # read techniques.csv and loop through each row
    data_frame = pd.read_csv("Techniques.csv")
    data_frame = data_frame[data_frame["is sub-technique"] == False]
    technique_id = []
    technique_name = []
    technique_description = []
    recommendation_id = []
    recommendation_title = []
    recommendation_description = []
    recommendation_score = []
    recommendation_reason = []
    recommendations_df = pd.read_csv("Recommendations.csv")
    for index, row in data_frame.iterrows():
        try:
            user_query = "Suggest recommendations for mitigating the below attack technique:\n"
            user_query += "Attack Name: " + row["name"] + "\n"
            user_query += "Attack Process: " + row["description"] + "\n"
            output = await mapping(user_query, recommendations_df)
            possible_recommendations = get_recommendations_filtered(recommendations_df, output)
            output2 = await validate(user_query, possible_recommendations)
            recommendations_list = json.loads(output2)["output_list"]
            for rec in recommendations_list:
                technique_id.append(row["ID"])
                technique_name.append(row["name"])
                technique_description.append(row["description"])
                rec_title = rec["title"]
                recommendation_title.append(rec_title)
                id, desc = get_recommendation_by_title(recommendations_df, rec_title)
                recommendation_id.append(id)
                recommendation_description.append(desc)
                recommendation_reason.append(rec["reason"])
                recommendation_score.append(rec["score"])
        except Exception as ex:
            print("Failed for Technique ID: ", row["ID"])
    # create a data frame using the lists
    data_frame = pd.DataFrame(
        list(
            zip(
                technique_id,
                technique_name,
                technique_description,
                recommendation_id,
                recommendation_title,
                recommendation_description,
                recommendation_score,
                recommendation_reason,
            )
        ),
        columns=[
            "Technique ID",
            "Technique Name",
            "Technique Description",
            "SCID",
            "Recommendation Title",
            "Recommendation Description",
            "Recommendation Score",
            "Explanation",
        ],
    )
    # save the data frame to a excel file
    data_frame.to_excel("output.xlsx", index=False)

In [11]:
async def main():
    data_frame = pd.read_csv("Techniques.csv")
    recommendations_df = pd.read_csv("Recommendations.csv")
    data_frame = data_frame[data_frame["is sub-technique"] == False]
    
    # 初始化存储结果的列表
    results = []
    
    for index, row in data_frame.iterrows():
        try:
            print(f"Processing Technique ID: {row['ID']}")  # 添加处理进度日志
            
            user_query = "Suggest recommendations for mitigating the below attack technique:\n"
            user_query += f"Attack Name: {row['name']}\n"
            user_query += f"Attack Process: {row['description']}\n"
            
            # 添加更详细的错误处理
            try:
                output = await mapping(user_query, recommendations_df)
            except Exception as e:
                print(f"Error in mapping for {row['ID']}: {str(e)}")
                continue
                
            try:
                possible_recommendations = get_recommendations_filtered(recommendations_df, output)
            except Exception as e:
                print(f"Error in filtering recommendations for {row['ID']}: {str(e)}")
                continue
                
            try:
                output2 = await validate(user_query, possible_recommendations)
                recommendations_list = json.loads(output2)["output_list"]
            except Exception as e:
                print(f"Error in validation for {row['ID']}: {str(e)}")
                continue
            
            # 处理每个推荐
            for rec in recommendations_list:
                rec_title = rec["title"]
                id, desc = get_recommendation_by_title(recommendations_df, rec_title)
                
                results.append({
                    "Technique ID": row["ID"],
                    "Technique Name": row["name"],
                    "Technique Description": row["description"],
                    "SCID": id,
                    "Recommendation Title": rec_title,
                    "Recommendation Description": desc,
                    "Recommendation Score": rec["score"],
                    "Explanation": rec["reason"]
                })
                
        except Exception as ex:
            print(f"Failed for Technique ID {row['ID']}: {str(ex)}")
            continue
    
    # 创建数据框
    if results:
        data_frame = pd.DataFrame(results)
        data_frame.to_excel("output.xlsx", index=False)
        print(f"Successfully processed {len(results)} recommendations")
    else:
        print("No results were generated")

In [12]:
!az login


[
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47",
    "id": "7ccdb8ae-4daf-4f0f-8019-e80665eb00d2",
    "isDefault": false,
    "managedByTenants": [],
    "name": "GCRProdEx1",
    "state": "Enabled",
    "tenantDefaultDomain": "microsoft.onmicrosoft.com",
    "tenantDisplayName": "Microsoft",
    "tenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47",
    "user": {
      "name": "v-xiangschen@microsoft.com",
      "type": "user"
    }
  },
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47",
    "id": "1214aacf-3446-4fb1-8803-94ba53a9e1cd",
    "isDefault": false,
    "managedByTenants": [],
    "name": "GCR Azure OpenAI 6",
    "state": "Enabled",
    "tenantDefaultDomain": "microsoft.onmicrosoft.com",
    "tenantDisplayName": "Microsoft",
    "tenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47",
    "user": {
      "name": "v-xiangschen@microsoft.com",
      "type": "user"
    }
  },
  {
    "cloudNa

In [13]:
await utils.startup()
await main()

Processing Technique ID: T1548
Error in mapping for T1548: RetryError[<Future at 0x15bd7852f10 state=finished raised InternalServerError>]
Processing Technique ID: T1134
Error in mapping for T1134: RetryError[<Future at 0x15bd785ce90 state=finished raised InternalServerError>]
Processing Technique ID: T1531


CancelledError: 